In [ ]:
import sys
import torch

print("Python executable:")
print(sys.executable)

print("\nPyTorch:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("\nGPU:")
    print(torch.cuda.get_device_name(0))

    print("\nVRAM:")
    print(
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Python executable:
F:\EarthMind\.venv\Scripts\python.exe

PyTorch:
2.5.1+cu121

CUDA available:
True

GPU:
NVIDIA RTX A6000

VRAM:
47.99 GB


In [ ]:
!pip install pandas pyarrow

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 4.2 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/9.9 MB 6.4 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.9 MB 8.8 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.9 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 12.0 MB/s  0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   --------- ------------------------------ 6.8/27.9 MB 32.3 MB/s eta 0:00:01
   ------------------------ --------------- 17.3/27.9 MB 42.0 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 50.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [pyarrow]
   ---------------------------------------- 0/2 [pyarrow]
   ---------------------------------------- 0/2 [pyarrow]
   ---------------------------------------- 0/2 [pyarrow]
   -------------

In [ ]:
import pandas as pd
print(pd.__version__)

3.0.6


In [ ]:
import pandas as pd

parquet_path = r"F:\EarthMind\BigEarthNet.txt\BigEarthNet.txt.parquet"

df = pd.read_parquet(parquet_path)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 9553962
Columns:
['ID', 's1_name', 'patch_id', 'input', 'output', 'type', 'category', 'split', 'latitude', 'longitude', 'country', 'season', 'climate_zone']


In [ ]:
print(df["split"].value_counts())

split
train         4674281
validation    2454690
test          2409962
bench           15029
Name: count, dtype: int64


In [ ]:
train = df[df["split"] == "train"]

print("Training annotation rows:", len(train))
print(
    "Unique S1-S2 pairs:",
    train[["s1_name", "patch_id"]].drop_duplicates().shape[0]
)

Training annotation rows: 4674281
Unique S1-S2 pairs: 229114


In [ ]:
sample_pair = train[["s1_name", "patch_id"]].iloc[0]

sample = train[
    (train["s1_name"] == sample_pair["s1_name"]) &
    (train["patch_id"] == sample_pair["patch_id"])
]

print("S1:", sample_pair["s1_name"])
print("S2:", sample_pair["patch_id"])

print(sample[["input", "output", "type", "category"]].to_string(index=False))

S1: S1B_IW_GRDH_1SDV_20170612T165809_33UUP_37_88
S2: S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_37_88
                                                                                                                                                                                                                   input                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [ ]:
# Check how many annotations each image pair has

pair_counts = (
    train.groupby(["s1_name", "patch_id"])
    .size()
)

print(pair_counts.describe())
print("\nMost common annotation counts:")
print(pair_counts.value_counts().head(20))

count    229114.000000
mean         20.401551
std           4.524627
min          14.000000
25%          18.000000
50%          20.000000
75%          24.000000
max          39.000000
dtype: float64

Most common annotation counts:
14    47843
18    27168
20    23115
22    18970
21    18001
23    16605
24    14372
25    13330
19     9751
26     9606
27     8372
16     5618
28     5227
29     3982
17     2629
30     2088
31     1348
32      526
33      326
34      147
Name: count, dtype: int64


In [ ]:
print(train["type"].value_counts())

type
binary          1785069
mcq             1602709
bounding box    1057389
captioning       229114
Name: count, dtype: int64


In [ ]:
print("\nCategory distribution:")
print(train["category"].value_counts())


Category distribution:
category
area            687342
count           687342
presence        687342
adjacency       591656
point           548320
reference       509069
None            229114
climate zone    229114
country         229114
season          229114
relative pos     46754
Name: count, dtype: int64


In [ ]:
print("\nSplit + type:")
print(
    pd.crosstab(
        train["split"],
        train["type"]
    )
)


Split + type:
type    binary  bounding box  captioning      mcq
split                                            
train  1785069       1057389      229114  1602709


In [ ]:
pd.crosstab(train["type"], train["category"])

category,None,adjacency,area,climate zone,count,country,point,presence,reference,relative pos,season
type,,,,,,,,,,,
binary,0,410385,458228,0,458228,0,0,458228,0,0,0
bounding box,0,0,0,0,0,0,548320,0,509069,0,0
captioning,229114,0,0,0,0,0,0,0,0,0,0
mcq,0,181271,229114,229114,229114,229114,0,229114,0,46754,229114


In [ ]:
type_category = pd.crosstab(
    train["type"],
    train["category"],
    normalize="index"
) * 100

print(type_category.round(2))

category       None  adjacency   area  climate zone  count  country  point  \
type                                                                         
binary          0.0      22.99  25.67           0.0  25.67      0.0   0.00   
bounding box    0.0       0.00   0.00           0.0   0.00      0.0  51.86   
captioning    100.0       0.00   0.00           0.0   0.00      0.0   0.00   
mcq             0.0      11.31  14.30          14.3  14.30     14.3   0.00   

category      presence  reference  relative pos  season  
type                                                     
binary           25.67       0.00          0.00     0.0  
bounding box      0.00      48.14          0.00     0.0  
captioning        0.00       0.00          0.00     0.0  
mcq              14.30       0.00          2.92    14.3  


In [ ]:
pairs = train.groupby(
    ["s1_name", "patch_id"],
    as_index=False
).first()

print("Unique training pairs:", len(pairs))

print("\nCountries:")
print(pairs["country"].value_counts().head(30))

print("\nSeasons:")
print(pairs["season"].value_counts())

print("\nClimate zones:")
print(pairs["climate_zone"].value_counts())

Unique training pairs: 229114

Countries:
country
Finland        72173
Portugal       42390
Serbia         36335
Austria        23817
Lithuania      22349
Ireland        20919
Belgium         6908
Luxembourg      1773
Switzerland     1633
Kosovo           817
Name: count, dtype: int64

Seasons:
season
Spring    78874
Fall      67515
Summer    46393
Winter    36332
Name: count, dtype: int64

Climate zones:
climate_zone
Cold, no dry season, warm summer         82592
Cold, no dry season, cold summer         41161
Temperate, no dry season, warm summer    35682
Temperate, dry summer, hot summer        35485
Temperate, no dry season, hot summer     26339
Temperate, dry summer, warm summer        6905
Cold, no dry season, hot summer            455
Cold, dry summer, warm summer              451
Arid, steppe, cold                          40
Temperate, no dry season, cold summer        4
Name: count, dtype: int64


In [ ]:
print("\nMissing values:")
print(
    pairs[["country", "season", "climate_zone", "latitude", "longitude"]]
    .isna()
    .sum()
)


Missing values:
country         0
season          0
climate_zone    0
latitude        0
longitude       0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import os
import json

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

TARGET_ANNOTATIONS = 10_000
TARGET_PAIRS = 500
SEED = 42

OUT_DIR = r"F:\EarthMind\SatQuery_10K"
os.makedirs(OUT_DIR, exist_ok=True)

# --------------------------------------------------
# TRAIN DATA
# --------------------------------------------------

train = df[df["split"] == "train"].copy()

print("Total train annotations:", len(train))

# --------------------------------------------------
# CREATE UNIQUE IMAGE-PAIR TABLE
# --------------------------------------------------

pair_counts = (
    train.groupby(["s1_name", "patch_id"], sort=False)
    .size()
    .rename("annotation_count")
    .reset_index()
)

pairs = (
    train[
        [
            "s1_name",
            "patch_id",
            "country",
            "season",
            "climate_zone",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates(["s1_name", "patch_id"])
    .merge(
        pair_counts,
        on=["s1_name", "patch_id"],
        how="left"
    )
)

print("Unique training pairs:", len(pairs))
print("Average annotations/pair:", pairs["annotation_count"].mean())

# --------------------------------------------------
# CREATE GEOGRAPHIC / CONTEXT STRATA
# --------------------------------------------------

pairs["stratum"] = (
    pairs["country"].astype(str)
    + " | "
    + pairs["season"].astype(str)
    + " | "
    + pairs["climate_zone"].astype(str)
)

stratum_counts = pairs["stratum"].value_counts()

print("\nNumber of strata:", len(stratum_counts))

# --------------------------------------------------
# ALLOCATE ~500 PAIRS USING SQRT WEIGHTING
#
# sqrt weighting:
# - doesn't let Finland dominate completely
# - doesn't over-represent tiny strata too heavily
# - still preserves diversity
# --------------------------------------------------

weights = np.sqrt(stratum_counts.values)
raw_alloc = (weights / weights.sum()) * TARGET_PAIRS

allocation = np.floor(raw_alloc).astype(int)

# Give at least 1 pair to every stratum when possible
allocation = np.maximum(allocation, 1)

# Cap by available pairs
allocation = np.minimum(allocation, stratum_counts.values)

# Fix total number of selected pairs
current_total = allocation.sum()

while current_total > TARGET_PAIRS:
    # Remove one from the largest allocations
    candidates = np.where(allocation > 1)[0]

    if len(candidates) == 0:
        break

    idx = candidates[np.argmax(allocation[candidates])]
    allocation[idx] -= 1
    current_total -= 1

while current_total < TARGET_PAIRS:
    # Add one to strata which still have unused pairs
    candidates = np.where(allocation < stratum_counts.values)[0]

    if len(candidates) == 0:
        break

    # Prefer strata with larger remaining population
    remaining = stratum_counts.values[candidates] - allocation[candidates]
    idx = candidates[np.argmax(remaining)]

    allocation[idx] += 1
    current_total += 1

allocation_map = dict(zip(stratum_counts.index, allocation))

print("Pairs planned for selection:", sum(allocation_map.values()))

# --------------------------------------------------
# SAMPLE PAIRS FROM EACH STRATUM
# --------------------------------------------------

selected_parts = []

for stratum, n in allocation_map.items():

    if n <= 0:
        continue

    group = pairs[pairs["stratum"] == stratum]

    sampled = group.sample(
        n=min(n, len(group)),
        random_state=SEED
    )

    selected_parts.append(sampled)

selected_pairs = pd.concat(
    selected_parts,
    ignore_index=True
)

# Shuffle final pair order
selected_pairs = selected_pairs.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("\nSelected unique pairs:", len(selected_pairs))

# --------------------------------------------------
# GET ALL ANNOTATIONS FOR SELECTED PAIRS
# --------------------------------------------------

selected_keys = selected_pairs[
    ["s1_name", "patch_id"]
].drop_duplicates()

selected_10k = train.merge(
    selected_keys,
    on=["s1_name", "patch_id"],
    how="inner"
)

print("Selected annotation rows:", len(selected_10k))

# --------------------------------------------------
# SAVE METADATA
# --------------------------------------------------

train_10k_path = os.path.join(
    OUT_DIR,
    "train_subset.parquet"
)

pairs_path = os.path.join(
    OUT_DIR,
    "selected_pairs.csv"
)

selected_10k.to_parquet(
    train_10k_path,
    index=False
)

selected_pairs.to_csv(
    pairs_path,
    index=False
)

# --------------------------------------------------
# STATISTICS
# --------------------------------------------------

stats = {
    "total_train_annotations": int(len(train)),
    "total_train_unique_pairs": int(len(pairs)),
    "selected_unique_pairs": int(len(selected_pairs)),
    "selected_annotations": int(len(selected_10k)),
    "average_annotations_per_selected_pair":
        float(selected_10k.groupby(
            ["s1_name", "patch_id"]
        ).size().mean())
}

with open(
    os.path.join(OUT_DIR, "selection_stats.json"),
    "w"
) as f:
    json.dump(stats, f, indent=4)

# --------------------------------------------------
# PRINT IMPORTANT RESULTS
# --------------------------------------------------

print("\n==============================")
print("SELECTION COMPLETE")
print("==============================")

print("Selected pairs:", len(selected_pairs))
print("Selected annotations:", len(selected_10k))

print("\nTYPE DISTRIBUTION:")
print(selected_10k["type"].value_counts())

print("\nCATEGORY DISTRIBUTION:")
print(selected_10k["category"].value_counts(dropna=False))

print("\nCOUNTRY DISTRIBUTION:")
print(selected_pairs["country"].value_counts())

print("\nSEASON DISTRIBUTION:")
print(selected_pairs["season"].value_counts())

print("\nCLIMATE DISTRIBUTION:")
print(selected_pairs["climate_zone"].value_counts())

print("\nFILES CREATED:")
print(train_10k_path)
print(pairs_path)
print(os.path.join(OUT_DIR, "selection_stats.json"))

Total train annotations: 4674281
Unique training pairs: 229114
Average annotations/pair: 20.40155119285596

Number of strata: 81
Pairs planned for selection: 500

Selected unique pairs: 500
Selected annotation rows: 10091

SELECTION COMPLETE
Selected pairs: 500
Selected annotations: 10091

TYPE DISTRIBUTION:
type
binary          3878
mcq             3491
bounding box    2222
captioning       500
Name: count, dtype: int64

CATEGORY DISTRIBUTION:
category
area            1500
count           1500
presence        1500
adjacency       1256
point           1158
reference       1064
None             500
climate zone     500
country          500
season           500
relative pos     113
Name: count, dtype: int64

COUNTRY DISTRIBUTION:
country
Finland        131
Serbia          93
Portugal        76
Austria         69
Ireland         43
Lithuania       41
Belgium         26
Luxembourg       8
Switzerland      8
Kosovo           5
Name: count, dtype: int64

SEASON DISTRIBUTION:
season
Spring   

In [ ]:
selected_pairs.head(10)[
    ["s1_name", "patch_id"]
]

,s1_name,patch_id
0,S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45,S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_0...
1,S1B_IW_GRDH_1SDV_20170907T043202_35VNL_10_27,S2A_MSIL2A_20170905T095031_N9999_R079_T35VNL_1...
2,S1A_IW_GRDH_1SDV_20180327T064326_29SNB_8_57,S2B_MSIL2A_20180326T112109_N9999_R037_T29SNB_0...
3,S1A_IW_GRDH_1SDV_20171203T183450_29SNB_71_2,S2A_MSIL2A_20171201T112431_N9999_R037_T29SNB_7...
4,S1A_IW_GRDH_1SDV_20170904T161304_34VDN_5_87,S2B_MSIL2A_20170906T101019_N9999_R022_T34VDN_0...
5,S1A_IW_GRDH_1SDV_20180206T153231_35VPK_86_39,S2B_MSIL2A_20180204T094159_N9999_R036_T35VPK_8...
6,S1B_IW_GRDH_1SDV_20171004T064231_29SNB_8_57,S2A_MSIL2A_20171002T112111_N9999_R037_T29SNB_0...
7,S1B_IW_GRDH_1SDV_20170613T160455_34VER_9_51,S2A_MSIL2A_20170613T101031_N9999_R022_T34VER_0...
8,S1A_IW_GRDH_1SDV_20170906T155749_34WFS_81_7,S2B_MSIL2A_20170906T101019_N9999_R022_T34WFS_8...
9,S1A_IW_GRDH_1SDV_20170831T164243_33UXP_31_83,S2B_MSIL2A_20170831T095029_N9999_R079_T33UXP_3...


In [ ]:
print("First S1:")
print(selected_pairs.iloc[0]["s1_name"])

print("\nFirst S2:")
print(selected_pairs.iloc[0]["patch_id"])

First S1:
S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45

First S2:
S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45


In [ ]:
import os
import re
import pandas as pd

# Selected pairs
csv_path = r"F:\EarthMind\SatQuery_10K\selected_pairs.csv"

selected_pairs = pd.read_csv(csv_path)

# First selected pair
row = selected_pairs.iloc[0]

s1_patch = row["s1_name"]
s2_patch = row["patch_id"]

# Parent scene name = remove final _row_col
s1_scene = re.sub(r"_\d+_\d+$", "", s1_patch)
s2_scene = re.sub(r"_\d+_\d+$", "", s2_patch)

s1_path = os.path.join(
    r"G:\My Drive\BigEarthNet-S1",
    s1_scene,
    s1_patch
)

s2_path = os.path.join(
    r"G:\My Drive\BigEarthNet-S2",
    s2_scene,
    s2_patch
)

print("S1 patch:")
print(s1_patch)
print("\nS1 path:")
print(s1_path)

print("\nS2 patch:")
print(s2_patch)
print("\nS2 path:")
print(s2_path)

print("\nS1 exists:", os.path.exists(s1_path))
print("S2 exists:", os.path.exists(s2_path))

S1 patch:
S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45

S1 path:
G:\My Drive\BigEarthNet-S1\S1B_IW_GRDH_1SDV_20170802T043751_34TFN\S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45

S2 patch:
S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45

S2 path:
G:\My Drive\BigEarthNet-S2\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45

S1 exists: False
S2 exists: False


In [ ]:
import os
import re
import pandas as pd

csv_path = r"F:\EarthMind\SatQuery_10K\selected_pairs.csv"
selected_pairs = pd.read_csv(csv_path)

row = selected_pairs.iloc[0]

s1_patch = row["s1_name"]
s2_patch = row["patch_id"]

s1_scene = re.sub(r"_\d+_\d+$", "", s1_patch)
s2_scene = re.sub(r"_\d+_\d+$", "", s2_patch)

s1_path = os.path.join(
    r"G:\My Drive\BigEarthNet-S1\BigEarthNet-S1",
    s1_scene,
    s1_patch
)

s2_path = os.path.join(
    r"G:\My Drive\BigEarthNet-S2\BigEarthNet-S2",
    s2_scene,
    s2_patch
)

print("S1 path:")
print(s1_path)
print("S1 exists:", os.path.exists(s1_path))

print("\nS2 path:")
print(s2_path)
print("S2 exists:", os.path.exists(s2_path))

S1 path:
G:\My Drive\BigEarthNet-S1\BigEarthNet-S1\S1B_IW_GRDH_1SDV_20170802T043751_34TFN\S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45
S1 exists: False

S2 path:
G:\My Drive\BigEarthNet-S2\BigEarthNet-S2\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45
S2 exists: False


In [ ]:
import os
import re
import pandas as pd

csv_path = r"F:\EarthMind\SatQuery_10K\selected_pairs.csv"
selected_pairs = pd.read_csv(csv_path)

row = selected_pairs.iloc[0]

s1_patch = row["s1_name"]
s2_patch = row["patch_id"]

# Correct S1 parent scene:
# remove _TILE_ROW_COL from the end
s1_scene = re.sub(r"_[A-Za-z0-9]{5}_\d+_\d+$", "", s1_patch)

# Correct S2 parent scene:
# remove _ROW_COL from the end
s2_scene = re.sub(r"_\d+_\d+$", "", s2_patch)

S1_ROOT = r"G:\My Drive\BigEarthNet-S1\BigEarthNet-S1"
S2_ROOT = r"G:\My Drive\BigEarthNet-S2\BigEarthNet-S2"

s1_path = os.path.join(
    S1_ROOT,
    s1_scene,
    s1_patch
)

s2_path = os.path.join(
    S2_ROOT,
    s2_scene,
    s2_patch
)

print("S1 patch :", s1_patch)
print("S1 scene :", s1_scene)
print("S1 path  :", s1_path)
print("S1 exists:", os.path.isdir(s1_path))

print("\nS2 patch :", s2_patch)
print("S2 scene :", s2_scene)
print("S2 path  :", s2_path)
print("S2 exists:", os.path.isdir(s2_path))

S1 patch : S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45
S1 scene : S1B_IW_GRDH_1SDV_20170802T043751
S1 path  : G:\My Drive\BigEarthNet-S1\BigEarthNet-S1\S1B_IW_GRDH_1SDV_20170802T043751\S1B_IW_GRDH_1SDV_20170802T043751_34TFN_0_45
S1 exists: False

S2 patch : S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45
S2 scene : S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN
S2 path  : G:\My Drive\BigEarthNet-S2\BigEarthNet-S2\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN\S2B_MSIL2A_20170802T092029_N9999_R093_T34TFN_00_45
S2 exists: False


In [ ]:
import os
import pandas as pd

S1_ROOT = r"G:\My Drive\BigEarthNet-S1\BigEarthNet-S1"
S2_ROOT = r"G:\My Drive\BigEarthNet-S2\BigEarthNet-S2"

# ---------------------------------------------------------
# 1. INDEX ACTUAL S1 PATCH FOLDERS
# ---------------------------------------------------------

print("Scanning S1 folders...")

s1_index = {}

for scene in os.scandir(S1_ROOT):
    if scene.is_dir():
        for patch in os.scandir(scene.path):
            if patch.is_dir():
                s1_index[patch.name] = patch.path

print("S1 patch folders found:", len(s1_index))


# ---------------------------------------------------------
# 2. INDEX ACTUAL S2 PATCH FOLDERS
# ---------------------------------------------------------

print("Scanning S2 folders...")

s2_index = {}

for scene in os.scandir(S2_ROOT):
    if scene.is_dir():
        for patch in os.scandir(scene.path):
            if patch.is_dir():
                s2_index[patch.name] = patch.path

print("S2 patch folders found:", len(s2_index))


# ---------------------------------------------------------
# 3. LOAD TRAIN DATA
# ---------------------------------------------------------

train = df[df["split"] == "train"].copy()

print("Train annotations:", len(train))


# ---------------------------------------------------------
# 4. EXACT PATCH-LEVEL MATCH
# ---------------------------------------------------------

train["s1_available"] = train["s1_name"].isin(s1_index)
train["s2_available"] = train["patch_id"].isin(s2_index)

available = train[
    train["s1_available"] &
    train["s2_available"]
].copy()


# ---------------------------------------------------------
# 5. RESULTS
# ---------------------------------------------------------

print("\n==============================")
print("RESULT")
print("==============================")

print("S1 patch folders:", len(s1_index))
print("S2 patch folders:", len(s2_index))

print("Annotations with BOTH S1 + S2:", len(available))

print(
    "Unique S1-S2 pairs with BOTH available:",
    available[["s1_name", "patch_id"]]
    .drop_duplicates()
    .shape[0]
)

Scanning S1 folders...
S1 patch folders found: 305572
Scanning S2 folders...
S2 patch folders found: 3775
Train annotations: 4674281

RESULT
S1 patch folders: 305572
S2 patch folders: 3775
Annotations with BOTH S1 + S2: 20782
Unique S1-S2 pairs with BOTH available: 910


In [ ]:
print("\nExample matched S1:")
print(next(iter(s1_index.items())))

print("\nExample matched S2:")
print(next(iter(s2_index.items())))


Example matched S1:
('S1A_IW_GRDH_1SDV_20170613T165043_33UUP_76_61', 'G:\\My Drive\\BigEarthNet-S1\\BigEarthNet-S1\\S1A_IW_GRDH_1SDV_20170613T165043\\S1A_IW_GRDH_1SDV_20170613T165043_33UUP_76_61')

Example matched S2:
('S2A_MSIL2A_20170613T101031_N9999_R022_T34VER_09_70', 'G:\\My Drive\\BigEarthNet-S2\\BigEarthNet-S2\\S2A_MSIL2A_20170613T101031_N9999_R022_T34VER\\S2A_MSIL2A_20170613T101031_N9999_R022_T34VER_09_70')


In [ ]:
import pandas as pd
import numpy as np
import os
import json

TARGET = 10_000
SEED = 42

# available = jo dataframe humne abhi banaya tha
# jisme S1 + S2 dono actual Drive par available hain

pair_counts = (
    available
    .groupby(["s1_name", "patch_id"])
    .size()
    .reset_index(name="annotation_count")
)

print("Available pairs:", len(pair_counts))
print("Available annotations:", len(available))

# Randomize pair order
pairs_shuffled = pair_counts.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# Select complete pairs until close to 10K
selected_rows = []
current = 0

for _, row in pairs_shuffled.iterrows():

    count = int(row["annotation_count"])

    if current + count <= TARGET:
        selected_rows.append(row)
        current += count

    if current >= TARGET - 40:
        break

selected_pairs_10k = pd.DataFrame(selected_rows)

# Get ALL annotations belonging to selected pairs
selected_10k = available.merge(
    selected_pairs_10k[["s1_name", "patch_id"]],
    on=["s1_name", "patch_id"],
    how="inner"
)

print("\n==============================")
print("10K SUBSET")
print("==============================")

print("Selected pairs:", len(selected_pairs_10k))
print("Selected annotations:", len(selected_10k))

print("\nTYPE:")
print(selected_10k["type"].value_counts())

print("\nCATEGORY:")
print(selected_10k["category"].value_counts(dropna=False))

# --------------------------------------------------
# SAVE
# --------------------------------------------------

OUT = r"F:\EarthMind\SatQuery_10K"

os.makedirs(OUT, exist_ok=True)

selected_10k_path = os.path.join(
    OUT,
    "train_10k.parquet"
)

selected_pairs_path = os.path.join(
    OUT,
    "selected_pairs_10k.csv"
)

selected_10k.to_parquet(
    selected_10k_path,
    index=False
)

selected_pairs_10k.to_csv(
    selected_pairs_path,
    index=False
)

print("\nSaved:")
print(selected_10k_path)
print(selected_pairs_path)

Available pairs: 910
Available annotations: 20782

10K SUBSET
Selected pairs: 437
Selected annotations: 9977

TYPE:
type
binary          3473
mcq             3141
bounding box    2926
captioning       437
Name: count, dtype: int64

CATEGORY:
category
point           1568
reference       1358
area            1311
count           1311
presence        1311
adjacency       1265
None             437
climate zone     437
country          437
season           437
relative pos     105
Name: count, dtype: int64

Saved:
F:\EarthMind\SatQuery_10K\train_10k.parquet
F:\EarthMind\SatQuery_10K\selected_pairs_10k.csv


In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# --------------------------------------------------
# PATHS
# --------------------------------------------------

OUT_ROOT = r"F:\EarthMind\SatQuery_10K"

S1_OUT = os.path.join(OUT_ROOT, "S1")
S2_OUT = os.path.join(OUT_ROOT, "S2")

os.makedirs(S1_OUT, exist_ok=True)
os.makedirs(S2_OUT, exist_ok=True)

# selected_pairs_10k already exists from previous cell
# s1_index and s2_index also already exist

pairs = selected_pairs_10k[["s1_name", "patch_id"]].drop_duplicates().copy()

print("Pairs to copy:", len(pairs))


# --------------------------------------------------
# COPY ONE PATCH
# --------------------------------------------------

def copy_patch(src, dst):
    os.makedirs(dst, exist_ok=True)

    copied = 0

    for entry in os.scandir(src):
        if entry.is_file():
            target = os.path.join(dst, entry.name)

            # Don't recopy an already completed file
            if os.path.exists(target):
                continue

            shutil.copy2(entry.path, target)
            copied += 1

    return copied


# --------------------------------------------------
# BUILD COPY JOBS
# --------------------------------------------------

jobs = []

for _, row in pairs.iterrows():

    s1_name = row["s1_name"]
    s2_name = row["patch_id"]

    s1_src = s1_index[s1_name]
    s2_src = s2_index[s2_name]

    s1_dst = os.path.join(S1_OUT, s1_name)
    s2_dst = os.path.join(S2_OUT, s2_name)

    jobs.append(("S1", s1_name, s1_src, s1_dst))
    jobs.append(("S2", s2_name, s2_src, s2_dst))


print("Total patch folders:", len(jobs))


# --------------------------------------------------
# PARALLEL COPY
# --------------------------------------------------

success = 0
failed = []

with ThreadPoolExecutor(max_workers=4) as executor:

    future_map = {
        executor.submit(copy_patch, src, dst):
        (modality, name, src, dst)
        for modality, name, src, dst in jobs
    }

    with tqdm(
        total=len(future_map),
        desc="Copying S1/S2 patches"
    ) as pbar:

        for future in as_completed(future_map):

            modality, name, src, dst = future_map[future]

            try:
                n = future.result()
                success += 1

                pbar.set_postfix(
                    last=f"{modality}:{name}",
                    files=n
                )

            except Exception as e:
                failed.append(
                    (modality, name, str(e))
                )

            pbar.update(1)


# --------------------------------------------------
# RESULT
# --------------------------------------------------

print("\n==============================")
print("COPY COMPLETE")
print("==============================")

print("Successful patches:", success)
print("Failed patches:", len(failed))

if failed:
    print("\nFirst failures:")
    for x in failed[:10]:
        print(x)

print("\nS1 output:")
print(S1_OUT)

print("\nS2 output:")
print(S2_OUT)

Pairs to copy: 437
Total patch folders: 874


Copying S1/S2 patches:   0%|          | 0/874 [00:00<?, ?it/s]


COPY COMPLETE
Successful patches: 741
Failed patches: 133

First failures:
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_78_32', '[Errno 28] No space left on device')
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_84_78', '[Errno 28] No space left on device')
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_75_84', '[Errno 28] No space left on device')
('S1', 'S1A_IW_GRDH_1SDV_20170613T165043_33UUP_84_59', '[Errno 28] No space left on device')
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_84_59', '[Errno 28] No space left on device')
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_78_23', '[Errno 28] No space left on device')
('S1', 'S1A_IW_GRDH_1SDV_20170613T165043_33UUP_78_33', '[Errno 28] No space left on device')
('S1', 'S1A_IW_GRDH_1SDV_20170613T165043_33UUP_78_23', '[Errno 28] No space left on device')
('S2', 'S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_78_33', '[Errno 28] No space left on device')
('S1', 'S1A_IW_GRDH_1SDV_20170613T165043_33UUP_89_3

In [ ]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

OUT_ROOT = r"F:\EarthMind\SatQuery_10K"

S1_OUT = os.path.join(OUT_ROOT, "S1")
S2_OUT = os.path.join(OUT_ROOT, "S2")

os.makedirs(S1_OUT, exist_ok=True)
os.makedirs(S2_OUT, exist_ok=True)

pairs = selected_pairs_10k[["s1_name", "patch_id"]].drop_duplicates()

jobs = []

for _, row in pairs.iterrows():
    s1_name = row["s1_name"]
    s2_name = row["patch_id"]

    jobs.append((
        "S1",
        s1_name,
        s1_index[s1_name],
        os.path.join(S1_OUT, s1_name)
    ))

    jobs.append((
        "S2",
        s2_name,
        s2_index[s2_name],
        os.path.join(S2_OUT, s2_name)
    ))


def copy_patch(job):
    modality, name, src_dir, dst_dir = job

    os.makedirs(dst_dir, exist_ok=True)

    copied = 0
    skipped = 0

    for entry in os.scandir(src_dir):

        if not entry.is_file():
            continue

        src = entry.path
        dst = os.path.join(dst_dir, entry.name)

        # Already completely copied
        if os.path.exists(dst):
            try:
                if os.path.getsize(dst) == os.path.getsize(src):
                    skipped += 1
                    continue
            except OSError:
                pass

            # Remove incomplete file
            try:
                os.remove(dst)
            except OSError:
                pass

        shutil.copy2(src, dst)

        # Verify copied file size
        if os.path.getsize(dst) != os.path.getsize(src):
            raise IOError(f"Size mismatch: {src}")

        copied += 1

    return modality, name, copied, skipped


print("Total patch folders:", len(jobs))

success = 0
failed = []

with ThreadPoolExecutor(max_workers=2) as executor:

    futures = {
        executor.submit(copy_patch, job): job
        for job in jobs
    }

    with tqdm(total=len(futures), desc="Copying patches") as pbar:

        for future in as_completed(futures):

            job = futures[future]

            try:
                modality, name, copied, skipped = future.result()

                success += 1

                pbar.set_postfix(
                    modality=modality,
                    copied=copied,
                    skipped=skipped
                )

            except Exception as e:
                modality, name, src, dst = job
                failed.append((modality, name, str(e)))

            pbar.update(1)


print("\n==============================")
print("COPY RESULT")
print("==============================")
print("Successful:", success)
print("Failed:", len(failed))

if failed:
    print("\nFirst failures:")
    for item in failed[:10]:
        print(item)

Total patch folders: 874


Copying patches:   0%|          | 0/874 [00:00<?, ?it/s]


COPY RESULT
Successful: 874
Failed: 0
